# Sepsis erken uyarı — Colab çalıştırma defteri
Önce **Çalışma zamanı → Çalışma zamanı türünü değiştir → T4 GPU** seç. Hücreleri sırayla çalıştır.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import torch
print('GPU:', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '-')

## 1) Depoyu çek ve kur

In [ ]:
%cd /content
!rm -rf sepsis-early-warning-hospital-shift
!git clone https://github.com/bahatrk/sepsis-early-warning-hospital-shift.git
%cd /content/sepsis-early-warning-hospital-shift
!pip install -q -e ".[dev]"
import os
assert os.path.isdir('src/sepsis/data'), "src/sepsis/data/ depoda YOK: .gitignore 'data/' satırı onu dışlamış. Önce depoyu düzeltip push et."
print('paket tamam')

## 2) Ham veriyi hazırla
`sepsis_s3` klasörünü (içinde `training_setA` ve `training_setB`) **zip'leyip** Drive'a `MyDrive/sepsis/sepsis_s3.zip` olarak yükle. Bir kez yapılır.

In [ ]:
!unzip -q -o /content/drive/MyDrive/sepsis/sepsis_s3.zip -d /content/
!ls /content/sepsis_s3 | head
!find /content/sepsis_s3 -name '*.psv' | wc -l   # 40336 olmalı

## 3) Colab yapılandırması
Ön işlenmiş veri hızlı olsun diye `/content`'te; modeller ve sonuçlar kalıcı olsun diye Drive'da.

In [ ]:
import yaml
cfg = yaml.safe_load(open('configs/default.yaml', encoding='utf-8'))
cfg['paths']['raw_dir'] = '/content/sepsis_s3'
cfg['paths']['processed_dir'] = '/content/processed'
cfg['paths']['artifacts_dir'] = '/content/drive/MyDrive/sepsis/artifacts'
cfg['paths']['results_dir'] = '/content/drive/MyDrive/sepsis/results'
cfg['gru']['threads'] = 2
cfg['grud']['threads'] = 2
cfg['train']['device'] = 'cuda'          # yeni modeller (06/07 betikleri) GPU'da çalışır; 03/04/05 CPU'dur
yaml.safe_dump(cfg, open('configs/colab.yaml','w', encoding='utf-8'), allow_unicode=True)
os.environ['SEPSIS_CONFIG'] = 'configs/colab.yaml'
os.makedirs(cfg['paths']['artifacts_dir'], exist_ok=True); os.makedirs(cfg['paths']['results_dir'], exist_ok=True)

## 4) Testler ve bölme kontrolü

In [ ]:
!pytest -q
!python scripts/01_make_split.py

## 5) Ön işleme (iki yön)

In [ ]:
!python scripts/02_preprocess.py "A->B"
!python scripts/02_preprocess.py "B->A"

## 6) Taban modeller

In [ ]:
!python scripts/03_train_lr.py "A->B"
!python scripts/03_train_lr.py "B->A"

In [ ]:
!python scripts/05_gru_seeds.py
!cat /content/drive/MyDrive/sepsis/results/gru_ozet.md

## 7) Hocanın istediği çıktılar: baz GRU ve kendi model (GRU-D tarzı), tek tohum
Adım adım: önce **bir epoch'un süresine bak** (çıktıda `... sn`). Süre çok uzunsa durdur ve bana yaz.

Önemli: `gru` ve `grud` **aynı cihazda (GPU)** çalıştırılır ki kıyas adil olsun. `gru` bu hücrede GPU ile yeniden eğitilir; CPU'daki eski 0,801 ± 0,007 sonuçlarından çok küçük sapma olabilir (normal).

In [ ]:
!python scripts/06_train_model.py "A->B" --model gru --seed 0

In [ ]:
!python scripts/06_train_model.py "A->B" --model grud --seed 0

### Çıktılara bak
Her çalıştırma `results/modeller/<model>_<yon>_seed<s>/` klasörüne şunları yazar: `egriler.png` (kayıp / accuracy / AUROC / AUPRC, eğitim ve doğrulama aynı grafikte), `roc_pr.png`, `karisiklik_*.png`, `rapor_*.md` (sınıf bazında precision/recall/F1, macro/weighted), `tablo.md` (eğitim/doğrulama/test yan yana) ve `kesisim.md` (kümeler arası kesişim).

In [ ]:
from IPython.display import Image, Markdown, display
d = '/content/drive/MyDrive/sepsis/results/modeller/grud_A_B_seed0'
display(Markdown(open(d + '/tablo.md', encoding='utf-8').read()))
display(Markdown(open(d + '/rapor_hedef_test.md', encoding='utf-8').read()))
display(Markdown(open(d + '/kesisim.md', encoding='utf-8').read()))
for f in ['egriler.png', 'roc_pr.png', 'karisiklik_hedef_test.png']:
    display(Image(d + '/' + f))

## 8) Ablasyon ve çoklu tohum (süre ölçüldükten ve sonuçlar makul çıktıktan sonra)
`07` betiği mevcut sonuçları atlar, eksik olanları eğitir ve `results/modeller/ozet.md` dosyasına ortalama ± std ile aşırı öğrenme ölçütünü yazar. Önce `--seeds 0 1 2` ile küçük başla.

In [ ]:
!python scripts/07_model_seeds.py --models gru gru_ozelliksiz grud grud_decaysiz --seeds 0 1 2
!cat /content/drive/MyDrive/sepsis/results/modeller/ozet.md

## 9) Sonuçları indir
Drive'daki `MyDrive/sepsis/results/modeller` klasörünü zip'leyip bilgisayarına indirir.

In [ ]:
!cd /content/drive/MyDrive/sepsis/results && zip -rq /content/modeller.zip modeller
from google.colab import files
files.download('/content/modeller.zip')

## 10) Aşırı öğrenme için düzenlileştirme taraması (yalnızca doğrulama kümesi)
`08` test kümelerini hiç yüklemez. Her ayar için eğitim−doğrulama AUPRC farkı ve doğrulama AUPRC'si 2 tohum üzerinden ortalanır. Önce hızlı olan `gru` ile başla (yaklaşık 10-15 dk/yön), sonra `grud` (yaklaşık 1 saat/yön). Sonuçlar `results/tarama/` içine yazılır; oturum kapanırsa hücreyi tekrar çalıştırmak yeterli değil, bitmiş ayarlar atlanmaz, o yüzden bir yönü bir hücrede çalıştır.

In [ ]:
!python scripts/08_regularization_sweep.py --model gru --yonler "A->B" "B->A" --seeds 0 1

In [ ]:
!python scripts/08_regularization_sweep.py --model grud --yonler "A->B" --seeds 0 1

In [ ]:
!python scripts/08_regularization_sweep.py --model grud --yonler "B->A" --seeds 0 1

In [ ]:
from IPython.display import Markdown, display
import glob
for f in sorted(glob.glob('/content/drive/MyDrive/sepsis/results/tarama/*.md')):
    display(Markdown(open(f, encoding='utf-8').read()))

In [ ]:
!cd /content/drive/MyDrive/sepsis/results && zip -rq /content/tarama.zip tarama
from google.colab import files
files.download('/content/tarama.zip')

## 11) Deney: eksiklik göstergesi bırakma (`gru_maskdrop`)
Bulgu (Bölüm 8): göstergeler kaynak hastanede yardım ediyor, başka hastanede etmiyor. Hipotez: eğitimde göstergeleri hasta bazında %50 olasılıkla sıfırlarsak model hastaneye özgü ölçüm alışkanlığına daha az bağımlı olur. Ayarlar `gru` ile aynı (config'teki varsayılanlar), tek fark bırakma; bu yüzden karşılaştırma adil. **Sonuç işe yaramayabilir; olduğu gibi raporlanır.**

In [ ]:
!python scripts/07_model_seeds.py --models gru_maskdrop --seeds 0 1 2

In [ ]:
!python scripts/07_model_seeds.py --models gru gru_ozelliksiz gru_maskdrop --seeds 0 1 2 --sadece-ozet
!cat /content/drive/MyDrive/sepsis/results/modeller/ozet.md

## 12) Nihai koşu: seçilen düzenlileştirme ile 5 tohum, tüm modeller
Tarama sonucu (`gru`, yalnızca doğrulama kümesi): `weight_decay` 1e-4 ile 1e-3 arasında fark yok; dropout 0.4 tek başına fark büyütüyor; asıl işe yarayan `gap_limit = 0.05` (eğitim−doğrulama AUPRC farkı 0.05'i aşan epoch seçilmez). Seçim yalnızca doğrulama bilgisine dayanır. **Aynı ayar tüm modellere uygulanır** (adil karşılaştırma). Sonuçlar eski sonuçların üzerine yazılmasın diye ayrı klasöre (`results_final`) gider; eski sonuçlar "önce" tablosu olarak kalır.

In [ ]:
import yaml, os
cfg = yaml.safe_load(open('configs/colab.yaml', encoding='utf-8'))
for k in ('gru', 'grud'):
    cfg[k]['gap_limit'] = 0.05
cfg['paths']['results_dir'] = '/content/drive/MyDrive/sepsis/results_final'
cfg['paths']['artifacts_dir'] = '/content/drive/MyDrive/sepsis/artifacts_final'
yaml.safe_dump(cfg, open('configs/colab_final.yaml', 'w', encoding='utf-8'), allow_unicode=True)
os.environ['SEPSIS_CONFIG'] = 'configs/colab_final.yaml'
os.makedirs(cfg['paths']['results_dir'], exist_ok=True); os.makedirs(cfg['paths']['artifacts_dir'], exist_ok=True)
print('gap_limit:', cfg['gru']['gap_limit'], cfg['grud']['gap_limit'], '| sonuçlar:', cfg['paths']['results_dir'])

Yaklaşık 1,5 saat sürer (5 model × 2 yön × 5 tohum). Oturum kapanırsa bu iki hücreyi yeniden çalıştır; biten koşular atlanır.

In [ ]:
!python scripts/07_model_seeds.py --models gru gru_ozelliksiz gru_maskdrop grud grud_decaysiz --seeds 0 1 2 3 4
!cat /content/drive/MyDrive/sepsis/results_final/modeller/ozet.md

In [ ]:
!cd /content/drive/MyDrive/sepsis && zip -rq /content/modeller_final.zip results_final/modeller
from google.colab import files
files.download('/content/modeller_final.zip')

## 13) Tarama (yalnızca doğrulama kümesi): weight decay ve veri artırma
Bölüm 1–5'ten sonra doğrudan buraya gel. **Önce `gru`** ile çalışılır (hızlı). Weight decay: AdamW'de adım başına küçülme `lr × wd` olduğundan 1e-4 etkisizdi; 1e-2 ve üstü deneniyor. Veri artırma yalnızca eğitimde uygulanır (jitter, magnitude scaling, hasta başına offset). `gap_limit=0.05` hepsinde sabit. Her hücre ~10-20 dk (iki yön, 2 tohum).

In [ ]:
!python scripts/08_regularization_sweep.py --model gru --dropout 0.2 --weight-decay 1e-4 1e-2 5e-2 1e-1 3e-1 --gap-limit 0.05 --etiket _wd

Weight decay taraması sonucu: **0.3** seçildi (3 tohum, doğrulama). Aşağıdaki üç hücrede `--weight-decay 0.3` yazılıdır.


In [ ]:
!python scripts/08_regularization_sweep.py --model gru --weight-decay 0.3 --gap-limit 0.05 --noise-std 0 0.05 0.1 0.2 --etiket _jitter

In [ ]:
!python scripts/08_regularization_sweep.py --model gru --weight-decay 0.3 --gap-limit 0.05 --scale-std 0 0.05 0.1 0.2 --etiket _scale

In [ ]:
!python scripts/08_regularization_sweep.py --model gru --weight-decay 0.3 --gap-limit 0.05 --shift-std 0 0.1 0.2 0.4 --etiket _offset

In [ ]:
from IPython.display import Markdown, display
import glob
for f in sorted(glob.glob('/content/drive/MyDrive/sepsis/results/tarama/gru_*_wd.md') + glob.glob('/content/drive/MyDrive/sepsis/results/tarama/gru_*_jitter.md') + glob.glob('/content/drive/MyDrive/sepsis/results/tarama/gru_*_scale.md') + glob.glob('/content/drive/MyDrive/sepsis/results/tarama/gru_*_offset.md')):
    print(f); display(Markdown(open(f, encoding='utf-8').read()))

In [ ]:
!cd /content/drive/MyDrive/sepsis/results && zip -rq /content/tarama2.zip tarama
from google.colab import files
files.download('/content/tarama2.zip')

## 14) Final koşu v2: wd=0.3, 5 model × 2 yön × 5 tohum + artırma kolları (yalnızca gru)
**Seçim kuralı (hedef teste bakmadan, önceden belirlendi):** ana ayar wd=0.3, dropout=0.2, gap_limit=0.05, artırma yok. Artırma kolları hedef testte **yan yana raporlanır, birinin 'seçilmesi' yoktur**: (a) artırma yok = ana koşu, (b) offset 0.2, (c) jitter 0.1 + scale 0.1 + offset 0.2. Kol (b) ve (c) yalnızca `gru`, 5 tohum, iki yön. Sonuçlar ayrı klasörlere yazılır. Bölüm 1–5 çalışmış olmalı.

### 14a) Ana koşu (5 model, artırma yok)

In [ ]:
import yaml, os
def hazirla(ad, noise=0.0, scale=0.0, shift=0.0, wd=0.3, processed=None, **ek):
    cfg = yaml.safe_load(open('configs/colab.yaml', encoding='utf-8'))
    for k in ('gru', 'grud'):
        cfg[k].update(gap_limit=0.05, weight_decay=wd, noise_std=noise, scale_std=scale, shift_std=shift, **ek)
    if processed: cfg['paths']['processed_dir'] = processed
    cfg['paths']['results_dir'] = f'/content/drive/MyDrive/sepsis/results_{ad}'
    cfg['paths']['artifacts_dir'] = f'/content/drive/MyDrive/sepsis/artifacts_{ad}'
    yol = f'configs/colab_{ad}.yaml'
    yaml.safe_dump(cfg, open(yol, 'w', encoding='utf-8'), allow_unicode=True)
    os.environ['SEPSIS_CONFIG'] = yol
    os.makedirs(cfg['paths']['results_dir'], exist_ok=True); os.makedirs(cfg['paths']['artifacts_dir'], exist_ok=True)
    print(ad, {k: cfg['gru'][k] for k in ('dropout','gap_limit','weight_decay','noise_std','scale_std','shift_std')})
hazirla('final2')

In [ ]:
!python scripts/07_model_seeds.py --models gru gru_ozelliksiz gru_maskdrop grud grud_decaysiz --seeds 0 1 2 3 4
!cat /content/drive/MyDrive/sepsis/results_final2/modeller/ozet.md

### 14b) Kol (b): offset 0.2 (yalnızca gru)  — ~20-30 dk

In [ ]:
hazirla('final2_offset', shift=0.2)

In [ ]:
!python scripts/07_model_seeds.py --models gru --seeds 0 1 2 3 4
!cat /content/drive/MyDrive/sepsis/results_final2_offset/modeller/ozet.md

### 14c) Kol (c): jitter 0.1 + scale 0.1 + offset 0.2 (yalnızca gru)  — ~20-30 dk

In [ ]:
hazirla('final2_hepsi', noise=0.1, scale=0.1, shift=0.2)

In [ ]:
!python scripts/07_model_seeds.py --models gru --seeds 0 1 2 3 4
!cat /content/drive/MyDrive/sepsis/results_final2_hepsi/modeller/ozet.md

### 14d) Hepsini indir

In [ ]:
!cd /content/drive/MyDrive/sepsis && zip -rq /content/modeller_final2_hepsi.zip results_final2/modeller results_final2_offset/modeller results_final2_hepsi/modeller
from google.colab import files
files.download('/content/modeller_final2_hepsi.zip')

## 15) Olasılık kalibrasyonu (Platt + etiketsiz EM öncül düzeltmesi), gru, 5 tohum
Yeni kod gerektirir: Bölüm 2'deki klonun `-b feature/deneyler` olması (güncellendi). Oturum yeniyse Bölüm 1–5'i çalıştır, sonra 14a'daki ilk hücreyi (`hazirla` fonksiyonu) çalıştır. Mevcut final koşular skor kaydetmediği için kalibrasyon yeniden eğitim ister: `gru`, wd=0.3, aynı ayar, yeni klasör. GPU kaynaklı küçük sayısal farklar olabilir. Her koşu ayrıca `skorlar.npz` kaydeder (sonraki bootstrap/eşleştirilmiş testler için). ~15-25 dk.

In [ ]:
hazirla('kalib')

In [ ]:
!python scripts/07_model_seeds.py --models gru --seeds 0 1 2 3 4
!cat /content/drive/MyDrive/sepsis/results_kalib/modeller/ozet.md

Kol (c) (jitter 0.1 + scale 0.1 + offset 0.2) için de kalibrasyon:

In [ ]:
hazirla('kalib_hepsi', noise=0.1, scale=0.1, shift=0.2)

In [ ]:
!python scripts/07_model_seeds.py --models gru --seeds 0 1 2 3 4
!cat /content/drive/MyDrive/sepsis/results_kalib_hepsi/modeller/ozet.md

In [ ]:
!cd /content/drive/MyDrive/sepsis && zip -rq /content/kalibrasyon.zip results_kalib/modeller results_kalib_hepsi/modeller
from google.colab import files
files.download('/content/kalibrasyon.zip')

## 16) Deney kolu: hastane bazlı normalizasyon (gru, 5 tohum)
**Önceden belirlenmiş tasarım:** hedef hastanenin kümeleri (adaptasyon, hedef test) için 34 ölçülen değişken, hedef hastanenin ETİKETSİZ `adaptasyon` hastalarının ortalama/std'siyle standartlaştırılır. Test hastaları ve etiketler kullanılmaz. Model, ayarlar ve tohumlar ana koşuyla aynıdır (wd 0.3, artırma yok); yalnızca hedef girdisi değişir. Oturum yeniyse Bölüm 1–5 ve 14a'nın ilk hücresi (`hazirla`) çalışmış olmalı, ham veri `/content/sepsis_s3` içinde olmalı. Kod dalı `feature/deneyler` (Bölüm 1'deki klon satırı güncellendi). ~25-35 dk.

In [ ]:
hazirla('hnorm', processed='/content/processed_hnorm')

In [ ]:
!python scripts/02_preprocess.py "A->B" --hedef-norm adaptasyon
!python scripts/02_preprocess.py "B->A" --hedef-norm adaptasyon

In [ ]:
from IPython.display import Markdown, display
for y in ('A_B', 'B_A'):
    display(Markdown(open(f'/content/processed_hnorm/{y}/kayma_tanisi.md', encoding='utf-8').read()))

In [ ]:
!python scripts/07_model_seeds.py --models gru --seeds 0 1 2 3 4
!cat /content/drive/MyDrive/sepsis/results_hnorm/modeller/ozet.md

In [ ]:
!cd /content && mkdir -p hn_tani && cp processed_hnorm/A_B/kayma_tanisi.md hn_tani/kayma_A_B.md && cp processed_hnorm/B_A/kayma_tanisi.md hn_tani/kayma_B_A.md
!cd /content/drive/MyDrive/sepsis && zip -rq /content/hnorm.zip results_hnorm/modeller /content/hn_tani
from google.colab import files
files.download('/content/hnorm.zip')

## 17) Ağaç tabanlı baseline: LightGBM + zengin pencere öznitelikleri (5 tohum, iki yön)
İki sürüm: `gbm` (eksiklik göstergeleri ve ölçüm sıklığı DAHİL) ve `gbm_davranissiz` (yalnızca değerler ve 6 saatlik pencere istatistikleri; hastanenin ölçüm alışkanlığını kodlayan sütunlar ÇIKARILMIŞ). Aynı bölme, aynı eşik/kalibrasyon raporları, aynı aşırı öğrenme ölçütü (eğitim−doğrulama AUPRC ≤ 0.05; kontrol noktası seçimi yalnızca eğitim/doğrulama ile). Bölüm 1–5 çalışmış olmalı (klon `-b feature/deneyler`, `pip install -e` LightGBM'i kurar), 14a'nın ilk hücresi (`hazirla`) çalışmış olmalı. İlk önce tek bir koşuyla süreyi gör. CPU kullanır.

In [ ]:
hazirla('gbm')

In [ ]:
!python scripts/06_train_model.py "A->B" --model gbm --seed 0

In [ ]:
!python scripts/07_model_seeds.py --models gbm gbm_davranissiz --seeds 0 1 2 3 4
!cat /content/drive/MyDrive/sepsis/results_gbm/modeller/ozet.md

In [ ]:
!cd /content/drive/MyDrive/sepsis && zip -rq /content/gbm.zip results_gbm/modeller
from google.colab import files
files.download('/content/gbm.zip')

## 18) LightGBM düzenlileştirme taraması (YALNIZCA eğitim/doğrulama)
Bölüm 17'de `gbm` ve `gbm_davranissiz` eğitim−doğrulama AUPRC ölçütünü (≤ 0.05) 5/5 koşuda aştı (ortalama fark ~0.058). Ağaçları daha sığ ve daha düzenli yapıp ölçütü sağlayan en iyi ayarı arıyoruz. Test ve adaptasyon kümeleri bu betikte HİÇ okunmaz. ~30-40 dk (iki yön, 8 ayar).

In [ ]:
!python scripts/09_gbm_sweep.py --yonler "A->B" "B->A" --num-leaves 4 8 --min-child 2000 5000 --lambda-l2 10 100 --etiket _a

## 19) LightGBM nihai koşu (taramada seçilen ayarlarla; 5 tohum, iki yön)
Tarama (Bölüm 18) sonucu `configs/default.yaml` içine işlendi (yaprak 4, min_child 5000, lambda_l2 10). Bölüm 17'nin eski sonuçları farklı ayarlarla üretildiği için YENİ klasöre (`results_gbm_final`) yazılır. Repo güncel dalda (`feature/deneyler`) olmalı: yeni `default.yaml` ile klonlayın (Bölüm 2).

In [ ]:
hazirla('gbm_final')

In [ ]:
!python scripts/07_model_seeds.py --models gbm gbm_davranissiz --seeds 0 1 2 3 4
!cat /content/drive/MyDrive/sepsis/results_gbm_final/modeller/ozet.md

In [ ]:
!cd /content/drive/MyDrive/sepsis && zip -rq /content/gbm_final.zip results_gbm_final/modeller
from google.colab import files
files.download('/content/gbm_final.zip')

## 20) LightGBM nihai koşu v2: seçimde eğitim AUPRC'si TÜM eğitim satırlarında
Bölüm 19'da seçim 200 bin satırlık alt örnekle yapıldığından tam eğitim kümesindeki fark bazı koşularda 0.05'i aştı. Düzeltilmiş kodla (repo güncel olmalı) yeni klasöre yeniden koşulur.

In [ ]:
hazirla('gbm_final2')

In [ ]:
!python scripts/07_model_seeds.py --models gbm gbm_davranissiz --seeds 0 1 2 3 4
!cat /content/drive/MyDrive/sepsis/results_gbm_final2/modeller/ozet.md

In [ ]:
!cd /content/drive/MyDrive/sepsis && zip -rq /content/gbm_final2.zip results_gbm_final2/modeller
from google.colab import files
files.download('/content/gbm_final2.zip')

## 21) Deep CORAL: etiketsiz hedef verisiyle eğitim sırasında temsil hizalama (gru, 5 tohum, iki yön)
**Önceden belirlenmiş tasarım:** `gru_coral` = ana gru (wd 0.3, dropout 0.2, gap_limit 0.05) + her adımda hedefin ETİKETSİZ `adaptasyon` hastalarından aynı sayıda hasta; gizli temsil kovaryansları arasında Deep CORAL kaybı λ ağırlıkla eklenir. Hedef etiketleri hiç kullanılmaz. Üç kol: λ = 10, 100, 1000 (CORAL kaybı ~1e-4 ölçeğinde olduğundan küçük λ etkisizdir; epoch başına `coral_kaybi` sonuc.json'da). Hepsi raporlanır; birincil kol, **kaynak doğrulama AUPRC'si** en yüksek olan λ'dır (hedef test seçimde kullanılmaz). GPU (T4) kullanın. Oturum yeniyse Bölüm 1–5 ve 14a'nın ilk hücresi. Toplam ~45-60 dk.

In [ ]:
hazirla('coral10', coral=10.0)

In [ ]:
!python scripts/07_model_seeds.py --models gru_coral --seeds 0 1 2 3 4
!cat /content/drive/MyDrive/sepsis/results_coral10/modeller/ozet.md | head -20

In [ ]:
hazirla('coral100', coral=100.0)

In [ ]:
!python scripts/07_model_seeds.py --models gru_coral --seeds 0 1 2 3 4
!cat /content/drive/MyDrive/sepsis/results_coral100/modeller/ozet.md | head -20

In [ ]:
hazirla('coral1000', coral=1000.0)

In [ ]:
!python scripts/07_model_seeds.py --models gru_coral --seeds 0 1 2 3 4
!cat /content/drive/MyDrive/sepsis/results_coral1000/modeller/ozet.md | head -20

In [ ]:
!cd /content/drive/MyDrive/sepsis && zip -rq /content/coral.zip results_coral10/modeller results_coral100/modeller results_coral1000/modeller
from google.colab import files
files.download('/content/coral.zip')

## 22) Ek baseline'lar (lr, rf, et, mlp) + önem ağırlıklandırma (gbm_davranissiz_iw)
CPU çalışma zamanı yeterli (GPU gerekmez); varsa **High RAM** açın. Oturum yeniyse Bölüm 1–5 ve 14a'nın ilk hücresi (`hazirla`). Önce 22a'daki tek koşularla süreyi görün; her model ayrı hücrede, sonuçlar aynı klasöre (`results_ek`) yazılır, yarım kalan hücre yeniden çalıştırılınca biten koşuları atlar. Öncelik sırası: önem ağırlıklandırma ve lr (hızlı) → et → rf → mlp (en yavaş).

In [ ]:
hazirla('ek')

### 22a) Süre testi (tek koşu)

In [ ]:
!python scripts/06_train_model.py "A->B" --model gbm_davranissiz_iw --seed 0
!python scripts/06_train_model.py "A->B" --model lr --seed 0

### 22b) 5 tohum, iki yön

In [ ]:
!python scripts/07_model_seeds.py --models gbm_davranissiz_iw lr --seeds 0 1 2 3 4

In [ ]:
!python scripts/07_model_seeds.py --models et --seeds 0 1 2 3 4

In [ ]:
!python scripts/07_model_seeds.py --models rf --seeds 0 1 2 3 4

In [ ]:
!python scripts/07_model_seeds.py --models mlp --seeds 0 1 2 3 4

In [ ]:
!cat /content/drive/MyDrive/sepsis/results_ek/modeller/ozet.md | head -40
!cd /content/drive/MyDrive/sepsis && zip -rq /content/ek.zip results_ek/modeller
from google.colab import files
files.download('/content/ek.zip')

## 23) MLP yeniden: daha güçlü düzenlileştirme (alpha 100/30/10/3, en fazla 10 epoch)
İlk MLP koşuları 0,05 ölçütünü 10/10 koşuda aştı. Eski MLP klasörleri yeni klasöre karışmasın diye ayrı klasör (`results_mlp2`). CPU yeterli.

In [ ]:
hazirla('mlp2')

In [ ]:
!python scripts/07_model_seeds.py --models mlp --seeds 0 1 2 3 4
!cat /content/drive/MyDrive/sepsis/results_mlp2/modeller/ozet.md | head -30
!cd /content/drive/MyDrive/sepsis && zip -rq /content/mlp2.zip results_mlp2/modeller
from google.colab import files
files.download('/content/mlp2.zip')